### Todas as arquiteturas vs. ARMAX/MOGWO (split 70/15/15)

Treina `Seq2SeqAttention` (`01_seq2seq_attention.ipynb`) e as quatro arquiteturas de `mimo_sys.architectures` (`MPNNForecaster`, `SymbolicGraphNetwork`, `StemGNN`, `Seq2SeqLatentGNN`) no split cronológico 70/15/15 do dataset completo, e compara com o ARMAX/MOGWO publicado (1 e 12 passos), seguindo o protocolo de `02_seq2seq_attention_splits_vs_armax.ipynb`. Análise em `docs/2_split_artigo_vs_dataset_completo.md`.

In [1]:
from pathlib import Path

import lightning as L
import mlflow
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler
from torch import nn
from torch.utils.data import DataLoader

from mimo_sys.architectures import (
    MPNNForecaster,
    Seq2SeqLatentGNN,
    StemGNN,
    SymbolicGraphNetwork,
)

SEED = 42
L.seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_PROCESSED = Path("../../data/processed")
DATA_EXTERNAL = Path("../../data/external")
MODELS_DIR = Path("../../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_COLS = ["freq_b1", "freq_b2", "freq_b3", "vazao_entrada"]
OUTPUT_COLS = ["vazao_distribuicao", "nivel_res", "pressao"]
FEATURE_COLS = INPUT_COLS + OUTPUT_COLS
N_NODES = len(FEATURE_COLS)
TARGET_IDX = [FEATURE_COLS.index(c) for c in OUTPUT_COLS]
N_TARGETS = len(TARGET_IDX)

INPUT_WINDOW = 24
OUTPUT_WINDOW = 12

/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/10/05 18:02:48 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Seed set to 42


### Dados

Mesmo split 70/15/15 e mesma janela (`INPUT_WINDOW=24`, `OUTPUT_WINDOW=12`) do notebook 02, para os resultados caírem na mesma tabela.

In [2]:
splits = {
    name: pd.read_parquet(DATA_PROCESSED / f"mimo_full_sample_{name}.parquet")
    for name in ("train", "val", "test")
}
full_df = pd.concat(splits.values(), ignore_index=True)
full_raw = full_df[FEATURE_COLS].to_numpy()
TRAIN_END = len(splits["train"])
VAL_END = TRAIN_END + len(splits["val"])
train_raw, val_raw = full_raw[:TRAIN_END], full_raw[TRAIN_END:VAL_END]
print(
    "treino/val/teste:",
    train_raw.shape,
    val_raw.shape,
    full_raw[VAL_END:].shape,
)

treino/val/teste: (1545, 7) (331, 7) (332, 7)


### Janelas e wrapper Lightning

`WindowDataset` monta `(x, y)` com `x` em `(input_window, n_nodes)` — o formato batch-first do contrato comum, sem o `permute` seq-first do Seq2Seq antigo. `GenericForecaster` adiciona a loss e as métricas de treino/validação/teste em cima de qualquer modelo do contrato; para `StemGNN` soma a `backcast_loss` (obrigatória, ver `architectures/README.md`).

In [3]:
class WindowDataset(torch.utils.data.Dataset):
    """Janelas (x, y) batch-first sobre uma série multivariada."""

    def __init__(self, data: np.ndarray, target_idx: list[int]) -> None:
        self.data = data
        self.target_idx = target_idx
        self.n = len(data) - INPUT_WINDOW - OUTPUT_WINDOW + 1

    def __len__(self) -> int:
        return self.n

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        mid = idx + INPUT_WINDOW
        x = self.data[idx:mid]
        y = self.data[mid : mid + OUTPUT_WINDOW][:, self.target_idx]
        return torch.FloatTensor(x), torch.FloatTensor(y)


def to_loader(data: np.ndarray, shuffle: bool) -> DataLoader:
    return DataLoader(
        WindowDataset(data, TARGET_IDX), batch_size=32, shuffle=shuffle
    )


class GenericForecaster(L.LightningModule):
    """Wrapper Lightning comum ao contrato `architectures`."""

    def __init__(self, model: nn.Module, lr: float = 1e-3) -> None:
        super().__init__()
        self.model = model
        self.criterion = nn.MSELoss()
        self.lr = lr

    def _step(self, batch, stage: str) -> torch.Tensor:
        x, y = batch
        pred = self.model(x)
        loss = self.criterion(pred, y)
        if isinstance(self.model, StemGNN):
            loss += self.model.backcast_loss(x)
        self.log(f"{stage}_loss", loss, prog_bar=(stage != "test"))
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")

    def test_step(self, batch, batch_idx):
        return self._step(batch, "test")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"},
        }


def build_model(name: str) -> nn.Module:
    kwargs = dict(
        n_nodes=N_NODES,
        input_window=INPUT_WINDOW,
        output_window=OUTPUT_WINDOW,
        target_idx=TARGET_IDX,
    )
    if name == "MPNN":
        return MPNNForecaster(hidden_size=64, message_dim=8, **kwargs)
    if name == "SymbolicGraphNetwork":
        return SymbolicGraphNetwork(hidden_size=64, message_dim=8, **kwargs)
    if name == "StemGNN":
        return StemGNN(hidden_size=64, **kwargs)
    if name == "Seq2SeqLatentGNN":
        return Seq2SeqLatentGNN(hidden_size=64, **kwargs)
    raise ValueError(name)


def train_model(name: str, train_loader, val_loader) -> GenericForecaster:
    L.seed_everything(SEED)
    lit_model = GenericForecaster(build_model(name))
    checkpoint = ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=1)
    trainer = L.Trainer(
        max_epochs=200,
        accelerator="auto",
        callbacks=[EarlyStopping(monitor="val_loss", patience=20), checkpoint],
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    with mlflow.start_run(run_name=name):
        mlflow.log_params({"model": name, "seed": SEED})
        trainer.fit(lit_model, train_loader, val_loader)
    best = GenericForecaster.load_from_checkpoint(
        checkpoint.best_model_path, model=build_model(name)
    )
    print(name, "melhor época:", trainer.current_epoch)
    return best.eval().to(device)

### ARMAX/MOGWO publicado (Tabela 3), sem reajuste

Mesmo código de `02_seq2seq_attention_splits_vs_armax.ipynb`: 1 passo (usa `y` medido nos lags) e 12 passos (free-run).

In [4]:
tabela3 = pd.read_csv(DATA_EXTERNAL / "tabela3_parametros_armax_papper.csv")
params = tabela3["MOGWO"].to_numpy()
U = full_df[INPUT_COLS].to_numpy().T
Y = full_df[OUTPUT_COLS].to_numpy().T


def armax_base(p, out, t, y, u):
    """Parte AR + exógena da predição da saída `out` em t (sem MA)."""
    base = out * 24
    others = [o for o in range(3) if o != out]
    a_self = p[base : base + 3]
    a_o1 = p[base + 3 : base + 6]
    a_o2 = p[base + 6 : base + 9]
    b_u = p[base + 9 : base + 21].reshape(4, 3)
    s = 0.0
    for k in range(3):
        s -= a_self[k] * y[out, t - 1 - k]
        s -= a_o1[k] * y[others[0], t - 1 - k]
        s -= a_o2[k] * y[others[1], t - 1 - k]
        s += b_u[:, k] @ u[:, t - 1 - k]
    return s


def armax_forecast(p, u, y, e, t0, horizon):
    """Free-run de `horizon` passos a partir de t0 (histórico medido)."""
    y_sim = y.copy()
    e_sim = e.copy()
    e_sim[:, t0:] = 0.0
    preds = np.zeros((3, horizon))
    for h in range(horizon):
        t = t0 + h
        for out in range(3):
            c_e = p[out * 24 + 21 : out * 24 + 24]
            correction = sum(c_e[k] * e_sim[out, t - 1 - k] for k in range(3))
            preds[out, h] = armax_base(p, out, t, y_sim, u) + correction
        y_sim[:, t] = preds[:, h]
    return preds


E = np.zeros_like(Y)
for out in range(3):
    for t in range(3, Y.shape[1]):
        E[out, t] = Y[out, t] - armax_base(params, out, t, Y, U)


def armax_predictions(start: int, stop: int):
    """ARMAX 1 passo e 12 passos nas amostras do forecast encadeado."""
    n = (stop - start) // OUTPUT_WINDOW * OUTPUT_WINDOW
    one_step = np.stack(
        [
            armax_forecast(params, U, Y, E, t, 1)[:, 0]
            for t in range(start, start + n)
        ]
    )
    twelve = np.concatenate(
        [
            armax_forecast(params, U, Y, E, t0, OUTPUT_WINDOW).T
            for t0 in range(start, start + n, OUTPUT_WINDOW)
        ]
    )
    return one_step, twelve

### Avaliação

`forecast` encadeia janelas não sobrepostas de `OUTPUT_WINDOW` passos no espaço normalizado e desfaz o `MinMaxScaler`. Mesmo protocolo das demais comparações.

In [5]:
def metrics_table(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    """MSE, MAE, R² e SMAPE por saída."""
    err = y_true - y_pred
    ss_tot = ((y_true - y_true.mean(axis=0)) ** 2).sum(axis=0)
    smape = 100 * np.mean(
        2 * np.abs(err) / (np.abs(y_true) + np.abs(y_pred) + 1e-12), axis=0
    )
    return pd.DataFrame(
        {
            "mse": (err**2).mean(axis=0),
            "mae": np.abs(err).mean(axis=0),
            "r2": 1 - (err**2).sum(axis=0) / ss_tot,
            "smape_%": smape,
        },
        index=OUTPUT_COLS,
    )


def forecast(model, scaler, start: int, stop: int) -> np.ndarray:
    """Forecast encadeado de 12 passos em [start, stop), escala original."""
    scaled = scaler.transform(full_raw)
    preds = []
    for t0 in range(start, stop - OUTPUT_WINDOW + 1, OUTPUT_WINDOW):
        window = torch.FloatTensor(scaled[t0 - INPUT_WINDOW : t0])
        window = window.unsqueeze(0).to(device)  # [1, input_window, nodes]
        with torch.no_grad():
            out = model(window)
        preds.append(out.squeeze(0).cpu().numpy())
    preds = np.concatenate(preds)
    return (
        preds * scaler.data_range_[TARGET_IDX] + scaler.data_min_[TARGET_IDX]
    )

### Carregar os pesos já treinados

Cada arquitetura foi treinada em um processo separado (ver nota em `train_model` acima) e salva em `models/<nome>_70_15_15.pth`. `Seq2SeqAttentionRef` é a mesma classe de `01_seq2seq_attention.ipynb`, reproduzida aqui só para poder carregar `seq2seq_attention_mimo_70_15_15.pth` (`02_seq2seq_attention_splits_vs_armax.ipynb`) sem reexecutar seu notebook.

In [6]:
class _Seq2SeqEncoder(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers, batch_first=False
        )

    def forward(self, x):
        outputs, (hidden, cell) = self.lstm(x)
        return outputs, hidden, cell


class _Seq2SeqDotAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.fc = nn.Linear(hidden_size * 2, hidden_size)

    def forward(self, hidden, encoder_outputs):
        hidden_last = hidden[-1]
        weights = torch.bmm(
            encoder_outputs.permute(1, 0, 2), hidden_last.unsqueeze(2)
        ).squeeze(2)
        weights = torch.softmax(weights, dim=1)
        context = torch.bmm(
            weights.unsqueeze(1), encoder_outputs.permute(1, 0, 2)
        ).squeeze(1)
        out = torch.tanh(self.fc(torch.cat((context, hidden_last), dim=1)))
        return out, weights


class _Seq2SeqDecoder(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers, batch_first=False
        )
        self.attention = _Seq2SeqDotAttention(hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        _, (hidden, cell) = self.lstm(x, (hidden, cell))
        attn_out, weights = self.attention(hidden, encoder_outputs)
        return self.fc(attn_out), hidden, cell, weights


class Seq2SeqAttentionRef(nn.Module):
    """Mesma arquitetura de `01_seq2seq_attention.ipynb` (seq-first)."""

    def __init__(
        self, input_size, hidden_size, output_size, output_window, num_layers=1
    ):
        super().__init__()
        self.encoder = _Seq2SeqEncoder(input_size, hidden_size, num_layers)
        self.decoder = _Seq2SeqDecoder(
            input_size, hidden_size, output_size, num_layers
        )
        self.output_window = output_window
        self.output_size = output_size

    def forward(self, source):
        seq_len, batch_size, _ = source.shape
        outputs = torch.zeros(
            self.output_window,
            batch_size,
            self.output_size,
            device=source.device,
        )
        encoder_outputs, hidden, cell = self.encoder(source)
        decoder_input = source[-1].unsqueeze(0)
        for t in range(self.output_window):
            decoder_output, hidden, cell, _ = self.decoder(
                decoder_input, hidden, cell, encoder_outputs
            )
            outputs[t] = decoder_output
            next_input = source[-1].clone()
            next_input[:, : self.output_size] = decoder_output
            decoder_input = next_input.unsqueeze(0)
        return outputs


def load_architecture(name: str) -> nn.Module:
    model = build_model(name)
    state = torch.load(
        MODELS_DIR / f"{name.lower()}_70_15_15.pth", map_location=device
    )
    model.load_state_dict(state)
    return model.eval().to(device)


scaler = MinMaxScaler().fit(train_raw)

seq2seq_ref = Seq2SeqAttentionRef(N_NODES, 64, N_TARGETS, OUTPUT_WINDOW, 2)
seq2seq_ref.load_state_dict(
    torch.load(
        MODELS_DIR / "seq2seq_attention_mimo_70_15_15.pth", map_location=device
    )
)
seq2seq_ref = seq2seq_ref.eval().to(device)


def forecast_seq2seq_ref(start: int, stop: int) -> np.ndarray:
    """Mesmo forecast encadeado de `forecast`, mas seq-first."""
    scaled = scaler.transform(full_raw)
    preds = []
    for t0 in range(start, stop - OUTPUT_WINDOW + 1, OUTPUT_WINDOW):
        window = torch.FloatTensor(scaled[t0 - INPUT_WINDOW : t0])
        window = window.unsqueeze(1).to(device)  # [input_window, 1, nodes]
        with torch.no_grad():
            out = seq2seq_ref(window)
        preds.append(out.squeeze(1).cpu().numpy())
    preds = np.concatenate(preds)
    return (
        preds * scaler.data_range_[TARGET_IDX] + scaler.data_min_[TARGET_IDX]
    )


ARCH_NAMES = ["MPNN", "SymbolicGraphNetwork", "StemGNN", "Seq2SeqLatentGNN"]
models = {name: load_architecture(name) for name in ARCH_NAMES}

### Comparação final

In [7]:
armax_1, armax_12 = armax_predictions(VAL_END, len(full_raw))
y_true = Y[:, VAL_END : VAL_END + len(armax_1)].T

rows = {
    "ARMAX 1 passo": metrics_table(y_true, armax_1),
    "ARMAX 12 passos": metrics_table(y_true, armax_12),
}
preds = {"Seq2SeqAttention": forecast_seq2seq_ref(VAL_END, len(full_raw))}
rows["Seq2SeqAttention"] = metrics_table(y_true, preds["Seq2SeqAttention"])
for name, model in models.items():
    pred = forecast(model, scaler, VAL_END, len(full_raw))
    preds[name] = pred
    rows[name] = metrics_table(y_true, pred)

results = pd.concat(rows, names=["modelo", "saida"])
results.round(4)

mse      mae      r2  smape_%
modelo               saida                                                 
ARMAX 1 passo        vazao_distribuicao   53.6091   4.9268  0.9147   8.0482
                     nivel_res             0.0034   0.0403  0.9318   1.7387
                     pressao               4.7234   1.1677  0.6709   5.3035
ARMAX 12 passos      vazao_distribuicao  270.3170  12.8180  0.5699  19.8845
                     nivel_res             0.0353   0.1345  0.2930   5.8150
                     pressao              12.2298   2.7366  0.1479  12.2252
Seq2SeqAttention     vazao_distribuicao   77.0344   6.5384  0.8774   9.7036
                     nivel_res             0.0282   0.1298  0.4348   5.4626
                     pressao               4.9822   1.4528  0.6529   6.5410
MPNN                 vazao_distribuicao   57.2911   5.7894  0.9088   8.6831
                     nivel_res             0.0376   0.1504  0.2476   6.3407
                     pressao               4.7851   1.5284  0.6666   6.8084
SymbolicGraphNetwork vazao_distribuicao   53.6485   5.5696  0.9146   8.4022
                     nivel_res             0.0370   0.1517  0.2599   6.3936
                     pressao               4.3992   1.4248  0.6935   6.3211
StemGNN              vazao_distribuicao   69.8334   6.3964  0.8889  10.1464
                     nivel_res             0.0307   0.1279  0.3856   5.4041
                     pressao               5.8234   1.7090  0.5942   7.7524
Seq2SeqLatentGNN     vazao_distribuicao   78.7510   6.9661  0.8747  10.5707
                     nivel_res             0.0348   0.1450  0.3036   6.1381
                     pressao               5.2648   1.5888  0.6332   7.2344

In [8]:
results["r2"].unstack("saida").round(3)

saida,vazao_distribuicao,nivel_res,pressao
modelo,,,
ARMAX 1 passo,0.915,0.932,0.671
ARMAX 12 passos,0.570,0.293,0.148
Seq2SeqAttention,0.877,0.435,0.653
MPNN,0.909,0.248,0.667
SymbolicGraphNetwork,0.915,0.260,0.693
StemGNN,0.889,0.386,0.594
Seq2SeqLatentGNN,0.875,0.304,0.633


In [9]:
labels = ["Vazão de distribuição (l/s)", "Nível (m)", "Pressão (mca)"]
idx = VAL_END + np.arange(len(y_true))
series = {
    "Medido": (y_true, "black"),
    "ARMAX 12 passos": (armax_12, "darkorange"),
}
series.update(
    {
        name: (p, c)
        for (name, p), c in zip(
            preds.items(),
            ["gray", "royalblue", "seagreen", "firebrick", "purple"],
        )
    }
)
fig = make_subplots(rows=3, cols=1, subplot_titles=labels)
for i in range(3):
    for name, (arr, color) in series.items():
        fig.add_trace(
            go.Scatter(
                x=idx,
                y=arr[:, i],
                name=name,
                line=dict(color=color, width=1),
                legendgroup=name,
                showlegend=(i == 0),
            ),
            row=i + 1,
            col=1,
        )
fig.update_layout(
    title_text="Teste 70/15/15: todas as arquiteturas vs. ARMAX/MOGWO",
    height=800,
    template="plotly_white",
    hovermode="x unified",
)
fig.show()